# 💧 LFM3-Omni - Spoken medical image Q&A with Transformers

This tutorial demonstrates how to fine-tune our omni model [`LiquidAI/LFM3-Omni-2B`](https://huggingface.co/LiquidAI/LFM3-Omni-2B)
on a task that combines **vision and speech**: a clinician looks at a medical image and **asks a question out loud**,
and the model picks the right answer.

We reuse the medical VQA dataset from our [LFM2.5-VL fine-tuning notebook](https://github.com/Liquid4All/cookbook/blob/main/finetuning/notebooks/sft_for_vision_language_model_with_trl.ipynb),
[simwit/omni-med-vqa-mini](https://huggingface.co/datasets/simwit/omni-med-vqa-mini) (CT, MRI, X-ray, ultrasound, dermoscopy,
fundus, microscopy and OCT images, with multiple-choice questions), and turn every question into speech with LFM3-Omni's own text-to-speech.

## 🎯 What You'll Find:
- **Data preparation** - synthesize spoken questions and build image + audio + text conversations
- **LoRA + SFT** - parameter-efficient fine-tuning with the 🤗 `Trainer`
- **Evaluation** - multiple-choice accuracy before and after fine-tuning

## 📋 Prerequisites:
- **GPU Runtime**: Select GPU in `Runtime` → `Change runtime type` (an A100 is recommended)
- **Hugging Face Account**: For accessing models and datasets

# 📦 Installation & Setup

First, let's install all the required packages:

In [ ]:
!pip install -qqq "transformers==5.20.2" "torchao==0.18.0" accelerate datasets peft pillow librosa soundfile --progress-bar off

Let's now verify the packages are installed correctly

In [ ]:
import os
import torch
import transformers
import peft
os.environ["WANDB_DISABLED"] = "true"

print(f"📦 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers version: {transformers.__version__}")
print(f"🧩 PEFT version: {peft.__version__}")

# Loading the model from Transformers 🤗

In [ ]:
from transformers import AutoModelForMultimodalLM, AutoProcessor

model_id = "LiquidAI/LFM3-Omni-2B"

print("📚 Loading processor...")
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

print("🧠 Loading model...")
model = AutoModelForMultimodalLM.from_pretrained(
    model_id, dtype=torch.bfloat16, device_map="cuda", trust_remote_code=True
)
codec = processor.audio_tokenizer.to(model.device)

print("\n✅ Model loaded successfully!")
print(f"🔢 Parameters: {model.num_parameters():,}")

# 📚 Load the dataset

Each sample has an image, a question, up to four options and the letter of the correct one.
We hold out 200 samples for evaluation.

In [ ]:
from datasets import load_dataset

raw_ds = load_dataset("simwit/omni-med-vqa-mini", split="test").shuffle(seed=42)
split = raw_ds.train_test_split(test_size=200, seed=42)
train_raw, eval_raw = split["train"], split["test"]

print("✅ Dataset loaded:")
print(f"   📚 Train samples: {len(train_raw)}")
print(f"   🧪 Eval samples: {len(eval_raw)}")

sample = train_raw[0]
display(sample["image"].convert("RGB").resize((224, 224)))
print(f"[{sample['modality']}] {sample['input']}\n✅ Answer: {sample['label']}")

# 🗣️ Turn the questions into speech

The dataset only has written questions, so we let LFM3-Omni read them aloud, alternating a female and a male voice.
Many questions repeat (1,146 unique questions for 2,000 samples), so we synthesize each one once.
This takes ~10-15 minutes on an A100.

In [ ]:
import librosa
import numpy as np
from IPython.display import Audio
from tqdm.auto import tqdm


@torch.inference_mode()
def speak(text, voice):
    inputs = processor.apply_chat_template(
        [
            {"role": "system", "content": f"Read this with a {voice} voice in American English."},
            {"role": "user", "content": text},
        ],
        add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt",
    ).to(model.device)
    out = model.generate(**inputs, max_new_tokens=400, generation_mode="audio", audio_temperature=0.4, audio_repetition_penalty=1.2)
    wav = codec.decode(out.audio_codes[:1, :, : out.audio_lengths[0]]).audio_values[0].float().cpu().numpy().squeeze()
    return librosa.resample(wav, orig_sr=24_000, target_sr=16_000)  # the audio encoder reads 16 kHz


torch.manual_seed(0)
questions = sorted(set(train_raw["question"]) | set(eval_raw["question"]))
spoken = {q: speak(q, "female" if i % 2 else "male") for i, q in enumerate(tqdm(questions, desc="Synthesizing questions"))}

print(f"🗣️ {len(spoken)} spoken questions, e.g. {sample['question']!r}")
display(Audio(spoken[sample["question"]], rate=16_000))

## Format the conversations

The image and the spoken question go in the user turn, followed by the written options. The target is the option letter.

In [ ]:
SYSTEM_PROMPT = "Answer questions about the provided image. Base every answer only on what can be seen. Respond with text."


def format_sample(sample, with_answer=True):
    options = "\n".join(f"({k}) {sample['option_' + k]}" for k in "ABCD" if sample["option_" + k])
    conversation = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": [
            {"type": "image", "image": sample["image"].convert("RGB")},
            {"type": "audio", "audio": spoken[sample["question"]]},
            {"type": "text", "text": f"Options:\n{options}\nAnswer with the letter of the correct option."},
        ]},
    ]
    if with_answer:
        conversation.append({"role": "assistant", "content": sample["label"]})
    return conversation


train_dataset = [format_sample(s) for s in train_raw]
print(processor.apply_chat_template(train_dataset[0], tokenize=False))

## Collate function

The processor renders a batch of conversations, loads the images and audio, and expands each `<|image|>` and `<|audio|>`
placeholder to its embeddings. We train only on the answer: every token up to and including the assistant header
(`<|im_start|>assistant\n<think></think>`) is masked with `-100`.

In [ ]:
ASSISTANT_HEADER = processor.tokenizer("<|im_start|>assistant\n<think></think>", add_special_tokens=False)["input_ids"]


def answer_start(ids):
    n = len(ASSISTANT_HEADER)
    return max(i for i in range(len(ids) - n + 1) if ids[i:i + n] == ASSISTANT_HEADER) + n


def collate_fn(conversations):
    batch = processor.apply_chat_template(conversations, tokenize=True, return_dict=True, return_tensors="pt", processor_kwargs={"padding": True})
    labels = torch.full_like(batch["input_ids"], -100)
    for i, ids in enumerate(batch["input_ids"].tolist()):
        start = answer_start(ids)
        labels[i, start:] = batch["input_ids"][i, start:]
    labels[batch["attention_mask"] == 0] = -100
    batch["labels"] = labels
    return batch


batch = collate_fn(train_dataset[:2])
print({k: tuple(v.shape) for k, v in batch.items()})
print("🎯 Supervised tokens:", [processor.tokenizer.decode(ids[lab != -100]) for ids, lab in zip(batch["input_ids"], batch["labels"])])

# 📏 Baseline accuracy

Let's measure how the model does before fine-tuning on the 200 held-out samples.

In [ ]:
import re


@torch.inference_mode()
def evaluate(model, show=3):
    correct = 0
    for k, sample in enumerate(tqdm(eval_raw, desc="Evaluating")):
        inputs = processor.apply_chat_template(
            format_sample(sample, with_answer=False),
            add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt",
        ).to(model.device)
        out = model.generate(**inputs, max_new_tokens=16, generation_mode="text")
        prediction = processor.decode(out.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        match = re.search(r"\b([ABCD])\b", prediction)
        correct += int(bool(match) and match.group(1) == sample["label"])
        if k < show:
            print(f"   {sample['question']!r} → {prediction!r} (answer {sample['label']})")
    return correct / len(eval_raw)


accuracy_before = evaluate(model)
print(f"📏 Accuracy before fine-tuning: {accuracy_before:.1%}")

## Wrap the model with PEFT

We add LoRA adapters to the language model: the attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`), the gated short convolutions (`in_proj`, `out_proj`), and the MLP (`w1`, `w2`, `w3`). The vision and audio encoders stay frozen. LFM3's attention layers also have a per-head output gate, `gate_proj`; adding it to the targets is a reasonable choice. Feel free to modify the `r` (rank) value:
- Higher → better approximation of full fine-tuning
- Lower → needs even less compute resources

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # optional: add "|gate_proj" to the group for LFM3's per-head attention gate
    target_modules=r"model\.language_model\..*\.(q_proj|k_proj|v_proj|o_proj|in_proj|out_proj|w1|w2|w3)",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 🚀 Launch Training

We use the 🤗 `Trainer`, which trains on the loss returned by the model. Our collate function builds the batches, so we keep all
columns (`remove_unused_columns=False`). One epoch over the 1,800 training samples takes a few minutes on an A100.

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="lfm3-omni-med-vqa",
    num_train_epochs=1,
    per_device_train_batch_size=2,  # images run at native resolution: keep this small on a 40 GB GPU
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    remove_unused_columns=False,
    report_to="none",
)

print("🏗️  Creating trainer...")
trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, data_collator=collate_fn)

print("\n🚀 Starting SFT training...")
trainer.train()
print("🎉 SFT training completed!")
print(f"💾 Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

# 📈 Accuracy after fine-tuning

In [ ]:
model.eval()
accuracy_after = evaluate(model)
print(f"📈 Accuracy  before: {accuracy_before:.1%}   after: {accuracy_after:.1%}")

# 💾 Save the model

Save the LoRA adapter, or merge it into the base weights to obtain a regular checkpoint.

In [ ]:
model.save_pretrained("./lfm3-omni-med-vqa-lora")
print("💾 Adapter saved to: ./lfm3-omni-med-vqa-lora")

In [ ]:
print("🔄 Merging LoRA weights...")
merged = model.merge_and_unload()
merged.save_pretrained("./lfm3-omni-med-vqa")
processor.save_pretrained("./lfm3-omni-med-vqa")
print("💾 Model saved to: ./lfm3-omni-med-vqa")

# 🚀 Next steps

- **Real recordings**: synthetic questions are a quick way to bootstrap; mix in recordings from your users' microphones for robustness to accents and noise.
- **Spoken answers**: to train the model to *answer* in speech, put the answer audio in the assistant turn; see [`lfm3_audio_tts_sft_with_transformers.ipynb`](lfm3_audio_tts_sft_with_transformers.ipynb) for how speech targets are built.